<a href="https://colab.research.google.com/github/IzzulGod/Sorachio-1B-Chat/blob/main/src/fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
from pathlib import Path

input_path = Path("/content/Data_Chunk_1.txt")
output_path = Path("/content/Train.jsonl")

with open(input_path, "r", encoding="utf-8") as file:
    lines = file.read().splitlines()

conversations = []
current_conv = []
state = None

for line in lines:
    line = line.strip()
    if line.startswith("User:"):
        if current_conv:
            conversations.append({"messages": current_conv})
            current_conv = []
        content = line[len("User:"):].strip()
        current_conv.append({"role": "user", "content": content})
        state = "user"
    elif line.startswith("Sorachio:"):
        content = line[len("Sorachio:"):].strip()
        current_conv.append({"role": "model", "content": content})
        state = "model"
    elif line == "":
        continue
    else:
        if state and current_conv:
            current_conv[-1]["content"] += "\n" + line

if current_conv:
    conversations.append({"messages": current_conv})

with open(output_path, "w", encoding="utf-8") as out:
    for convo in conversations:
        json.dump(convo, out, ensure_ascii=False)
        out.write("\n")

print(f"Dataset berhasil dikonversi ke format Gemma Chat. Simpan di: {output_path}")


Dataset berhasil dikonversi ke format Gemma Chat. Simpan di: /content/Train.jsonl


In [1]:
!pip install transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype="auto"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [4]:
from peft import get_peft_model, LoraConfig, TaskType
import torch

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,981,888 || all params: 1,002,867,840 || trainable%: 0.2973


In [5]:
from datasets import Dataset
import json

with open("/content/Train.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

data = [json.loads(line) for line in lines]

dataset = Dataset.from_list(data)

for i in range(min(3, len(dataset))):
    print(json.dumps(dataset[i], indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "content": "Halo",
      "role": "user"
    },
    {
      "content": "Halo juga! Senang banget bisa ngobrol sama kamu hari ini 😊 Ada yang bisa aku bantu?",
      "role": "model"
    }
  ]
}
{
  "messages": [
    {
      "content": "Siapa kamu?",
      "role": "user"
    },
    {
      "content": "Aku Sorachio, asisten AI yang ramah dan siap nemenin kamu ngobrol atau bantu cari info apa pun. Mau nanya-nanya, curhat, atau sekadar ngobrol santai juga boleh banget 😄",
      "role": "model"
    }
  ]
}
{
  "messages": [
    {
      "content": "Siapa yang menciptakanmu?",
      "role": "user"
    },
    {
      "content": "Aku dikembangkan oleh tim 1dle Labs — mereka fokus bikin AI yang bukan cuma pintar, tapi juga terasa dekat dan manusiawi. Tujuannya biar interaksi kayak gini nggak kaku dan bisa bikin kamu nyaman.",
      "role": "model"
    }
  ]
}


In [6]:
from transformers import DataCollatorForLanguageModeling

def tokenize(example):
    return tokenizer(
        tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=True
        ),
        truncation=True,
        max_length=385
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

lengths = [
    len(tokenizer.apply_chat_template(x["messages"], tokenize=True))
    for x in dataset
]
print("Max length:", max(lengths))
print("Avg length:", sum(lengths) // len(lengths))



Map:   0%|          | 0/655 [00:00<?, ? examples/s]

Max length: 668
Avg length: 93


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    warmup_ratio=0.03,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_dir="logs",
    logging_steps=20,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/tmp/ipython-input-7-263597711.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
20,5.340900
40,3.281900
60,2.771800
80,2.636200
100,2.376600
120,2.257200
140,2.275700
160,2.271100
180,2.090400
200,2.138500


TrainOutput(global_step=246, training_loss=2.6343929864526765, metrics={'train_runtime': 180.8773, 'train_samples_per_second': 10.864, 'train_steps_per_second': 1.36, 'total_flos': 1155927167603712.0, 'train_loss': 2.6343929864526765, 'epoch': 3.0})

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    attn_implementation="eager"
)

lora_model = PeftModel.from_pretrained(
    base_model,
    "/content/output/checkpoint-246",
    is_trainable=False
)

merged_model = lora_model.merge_and_unload()

save_path = "/content/drive/MyDrive/sorachio-merged"
merged_model.save_pretrained(save_path, safe_serialization=True)
tokenizer.save_pretrained(save_path)


In [9]:
from shutil import copytree

source_adapter_path = "/content/output/checkpoint-246"
destination_path = "/content/drive/MyDrive/sorachio-lora-adapter"

copytree(source_adapter_path, destination_path, dirs_exist_ok=True)


'/content/drive/MyDrive/sorachio-lora-adapter'